# Entropy Maximization via HUANet

Implementation of the Entropy Maximization example from the paper

[HUANet: Hard-Constrained Unrolled ADMM for Constrained Convex Optimization](https://arxiv.org/pdf/2604.13179v1).

*We'll assume basic familiarity with convex optimization, and refer readers to [[Boyd and Vandenberghe, 2004](https://web.stanford.edu/~boyd/cvxbook/)] for a longer overview.*

**Hard-Constrained Unrolled ADMM Network** (HUANet) is a learning-to-optimize framework that unrolls the iterations of the Alternating Direction Method of Multipliers (ADMM) into a trainable neural network for solving constrained convex optimization problems.
Existing end-to-end learning methods operate as black-box mappings from parameters to solutions, often lacking explicit optimality principles and failing to enforce
constraints.
To address this limitation, we unroll ADMM and embed a hard-constrained neural network at each iteration
to accelerate the algorithm, where equality constraints are
enforced via a differentiable correction stage at the network
output.
Furthermore, we incorporate first-order optimality conditions as soft constraints during training to promote the
convergence of the proposed unrolled algorithm.

# Problem formulation
For the purposes of this tutorial, which focuses on the practical implementation and application of HUANet, we will consider the Entropy Maximization optimization problem.

\begin{align}
\max_{x_i > 0} \quad & -\sum_{i=1}^{n_x} x_i \log(x_i) \\
\text{s.t.} \quad & C x \le d_\lambda, \\
& \mathbf{1}^\top x = 1.
\end{align}
where $x = [x_1, x_2, \ldots, x_{n_x}]^\top \in \mathbb{R}^{n_x}$ is the decision vector representing a probability distribution,
the matrix $C \in \mathbb{R}^{{n_{in}}\times {n_x}}$ is sampled as $(C)_{ij} \sim N(0,1)$,
and the right-hand side parameter $d_\lambda \in \mathbb{R}^{n_{in}}$ is generated by $d_\lambda = Cv + \epsilon$ with $v_i \sim U(0,1)$ and $\epsilon_i \sim N(0, 0.1)$.
Here we consider $d_\lambda$ as the parameter of the inequality constraints.



# Setup
Prepare the Python environment for the whole notebook by importing packages.

We turn on ```jax_enable_x64```flag to switch to 64 bit numbers for high accuracy.
By setting ```XLA_FLAGS``` befor importing JAX and other relevant libraries, we create virtual CPU devices to enable parallel computation.


In [ ]:
## Enable this cell when using Google Colab
# !git clone https://github.com/trinhtran1120/L2O_tutorial.git

In [2]:
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=20"

import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Tuple

import cvxpy as cp
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from jax import random, vmap
from flax import linen as nn
from functools import partial

jax.config.update("jax_enable_x64", True)

# Generating problem instances

We generate an input dataset as follows:

1. Sample a fixed inequality matrix $C$ from a standard normal distribution.
2. Generate a candidate vector $x_{\mathrm{feas}}$ ensuring all entries are positive and sum to one.
3. Set $d_\lambda = C x_{\mathrm{feas}} + \epsilon,$ where $\epsilon \sim {U}(0, 0.1)$ elementwise.

The generated dataset is saved as compressed `.npz` files.


Next, we set the core configuration for the entropy experiment.


In [ ]:
# Problem configuration
n_var = 10    # dimension of the decision vector x
n_in  = 5
n_eq  = 5

# Datasetsize
seed      = 2026
n_samples = 2000  # number of problem instances

# Output folder
## Use this path when using IDE
output_root = Path("data") 

## Use this path when using Google Colab
# output_root = Path(("/content/L2O_tutorial/HUANet"))


After setting the dimensions, the `generate_data` function generates a single valid problem instance (right-hand-side vector $d_\lambda$).


In [4]:
def generate_data(key, n_var, C):
    x_raw  = jax.random.uniform(key, (n_var,), minval=0.001, maxval=1.0)
    x_feas = jax.nn.softmax(x_raw)
    lam    = C @ x_feas + jax.random.uniform(key, (C.shape[0],), minval=0.0, maxval=0.1)
    return lam

Now let's create a dataset.
The matrix `C` is fixed for the whole experiment, meaning all generated optimization problems share the same left-hand-side inequality matrix.


In [5]:
key   = jax.random.PRNGKey(seed)
key_C, key_lam    = random.split(key)

C     = jax.random.normal(key_C, (n_in, n_var))

keys  =  random.split(key_lam, n_samples)
lam   = vmap(lambda k: generate_data(k, C=C, n_var=n_var))(keys)

After generating the samples, save the generated dataset.


In [6]:
dataset_file = f"datasets_demo_{n_samples}.npz"
dataset_path = os.path.join(output_root, dataset_file)
np.savez_compressed(dataset_path, lam=np.array(lam), C=np.array(C), N_SAMPLES=n_samples, N_VAR=n_var, N_EQ=n_eq,N_IN=n_in)
print(f"Successfully saved dataset: {dataset_path}")

Successfully saved dataset: data/datasets_demo_2000.npz


# Training HUANet

With the dataset prepared, this section trains HUANet for the entropy problem. The paper's main idea is to build a learned solver that keeps the structure of ADMM instead of replacing the entire solver with a black-box neural network.

HUANet unfolds the ADMM iterations into $N$ sequential neural layers, where each layer corresponds to an ADMM iteration and executes an identical neural mapping $F_{\theta_p}$. Each layer $F_{\theta_{p}}$ maps an iteration index $k$ using two primary building blocks:

*   Primal Network ($HNN_{\theta_{p}}$): Formed by a Multi-Layer Perceptron (MLP) that maps the current problem context into a prior, unconstrained estimate of the primal-slack variable pair
\begin{equation}
(q^{k}, \lambda) \mapsto \overline{y}^{k+1} = \begin{bmatrix} \overline{x}^{k+1} \\ \overline{s}^{k+1} \end{bmatrix}
\end{equation}
where $q^k = w^k - {\rho}^{-1}v^k$.
*   Correction Stage: An end-layer projection that mathematically forces the unconstrained estimates onto the affine subspace.

While the architecture guarantees primal feasibility via the correction stage, convergence to a mathematically optimal solution requires balancing first-order Karush-Kuhn-Tucker (KKT) optimality criteria.
To keep the pipeline fully differentiable and self-supervised, a secondary network ($MLP_{\theta_{d}}$) is deployed to estimate the true Lagrange multipliers ($z^{k+1}$) of the primal updates.

HUANet is trained in a self-supervised manner.
For a mini-batch of $S$ sampled parameter instances, HUANet coordinates its joint optimization parameters $(\theta_{p}, \theta_{d})$ by minimizing a composite penalty function:
\begin{equation}
{L}_{\lambda}(\theta_{p}, \theta_{d}) = \frac{1}{S}\sum_{i=1}^{S} \left( h_{\lambda}(\hat{y}_{i}^{N}) + \gamma_{r}\sum_{k=1}^{N}\|r_{i}^{k}\|_{2}^{2} \right)
\end{equation}
where terminal cost $h_{\lambda}(\hat{y}_{i}^{N}) = f_{\lambda}(\hat{x}_{i}^{N}) + \gamma_{s}\|\max(0, -\hat{s}_{i}^{N})\|_{2}^{2}$ enforces objective optimality and inequality-violation penalties at the last neural layer.
Residual cost $\gamma_{r}\sum_{k=1}^{N}\|r_{i}^{k}\|_{2}^{2}$ penalizes departures from the mathematical stationary trajectory throughout execution to improve overall solver stability.


From this training objective, we first set the training hyperparameters.


In [7]:
train_percent = 0.8             # Fraction of generated dataset used for training
val_percent = 0.1               # Fraction used for validation
test_percent = 0.1              # Fraction used for testing
batch_size = 32                # Mini-batch size
n_epochs = 5000                  # Number of training epochs/iterations
n_admm = 20                     # Number of unrolled ADMM steps
rho = 2.0                       # ADMM penalty parameter
hidden_layers = (32, 32)        # HUANet hidden layer sizes
eps = 1.0                       # Weight of KKT residual term
gamma = 1.0e3                   # Weight of primal feasibility violation

Next, HUANet first predicts an unconstrained vector $\bar y = [\bar x^\top, \bar s^\top]^\top.$
The correction stage then projects $\bar y$ onto the affine constraints. For the general HUANet form, these constraints are

$$
A_\lambda x = b_\lambda, \qquad C_\lambda x + s = d_\lambda.
$$

Here, $A_\lambda$ is the equality-constraint matrix. In this entropy example, the equality constraint is the simplex condition $\mathbf{1}^\top x = 1$, so $A_\lambda = \mathbf{1}^\top$ and $b_\lambda = 1$.
The correction projection is represented as a constrained optimization problem

$$
\hat y = \arg\min_y \frac{1}{2}\|y - \bar y\|_2^2
\quad \text{s.t.} \quad
E_\lambda y = \eta_\lambda,
$$

where

$$
E_\lambda =
\begin{bmatrix}
A_\lambda & 0 \\
C_\lambda & I
\end{bmatrix},
\qquad
\eta_\lambda =
\begin{bmatrix}
b_\lambda \\
d_\lambda
\end{bmatrix}.
$$

The KKT conditions for this projection are

$$
\hat y - \bar y + E_\lambda^\top \nu = 0,
\qquad
E_\lambda \hat y = \eta_\lambda.
$$

Solving these equations gives the closed-form correction

$$
\hat y = \bar y - E_\lambda^\top(E_\lambda E_\lambda^\top)^{-1}(E_\lambda\bar y - \eta_\lambda).
$$

The next function builds `E` and precomputes the reusable matrix $E_\lambda^\top(E_\lambda E_\lambda^\top)^{-1}$ so later correction steps only need fast matrix-vector operations.


In [8]:
def precompute_projection(A: jnp.ndarray, C: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
    n_eq = A.shape[0]
    n_in = C.shape[0]

    E = jnp.block([
        [A, jnp.zeros((n_eq, n_in))],
        [C, jnp.eye(n_in)]
        ])
    EtE_inv = jnp.linalg.solve(E @ E.T, E).T

    return E, EtE_inv

Using the precomputed matrix above, we apply the projection formula explained above.
It concatenates the raw network outputs $(\bar x, \bar s)$ into $\bar y$, forms the right-hand side $\eta = [b^\top, d^\top]^\top$, then computes the corrected vector $\hat y$ using the precomputed matrices `E` and `EtE_inv`.
Finally, it splits $\hat y$ back into $(\hat x, \hat s)$, which now satisfies the affine constraints used by the correction stage.


In [9]:
def correction(x_bar:jnp.ndarray, s_bar:jnp.ndarray, b:jnp.ndarray, d:jnp.ndarray, E:jnp.ndarray, EtE_inv: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray]:
    n_var = x_bar.shape[0]
    y_bar = jnp.concatenate([x_bar, s_bar], axis=0)
    eta   = jnp.concatenate([b, d], axis=0)
    y_hat = y_bar - EtE_inv @ (E @ y_bar - eta)

    return y_hat[:n_var], y_hat[n_var:]

Next, `PrimeNet` is the MLP inside the hard-constrained neural network $\mathbf{HNN}_{\theta_p}$.
The output $(\bar x, \bar s)$ of `PrimeNet` is a prior estimate.
In `feasibility` mode, used by this entropy example, `PrimeNet` outputs unconstrained scores for $x$; `HUANet` applies `softmax` to make them positive and sum to one. The `sigmoid` branch is only used in `correction` mode, where it keeps the preliminary estimates $(\bar x, \bar s)$ bounded before the projection step.


In [10]:
class PrimeNet(nn.Module):
    hidden_layers: tuple[int, ...]
    n_var:  int
    n_in: int
    mode: str = "correction"   # "correction" | "feasibility"

    def setup(self) -> None:
        # Initialization
        self.kernel_init = nn.initializers.xavier_uniform()
        self.bias_init   = nn.initializers.zeros_init()

    @nn.compact
    def __call__(self, nn_input: jnp.ndarray) -> Tuple[jnp.ndarray, jnp.ndarray] | jnp.ndarray:
        x = nn_input
        for features in self.hidden_layers:
            x = nn.Dense(features, kernel_init=self.kernel_init, bias_init=self.bias_init)(x)
            x = nn.gelu(x)

        raw = nn.Dense(self.n_var + self.n_in, kernel_init=self.kernel_init, bias_init=self.bias_init)(x)

        if self.mode == "feasibility":
            return raw[:, :self.n_var]
        else:
            out   = nn.sigmoid(raw)
            x_bar = out[:, :self.n_var]
            s_bar = out[:, self.n_var:]
            return x_bar, s_bar

After `PrimeNet`, `DualNet` is the MLP neural network used to estimate the Lagrange multiplier in the primal ADMM subproblem. Specifically, it learns

$$
z^{k+1} \approx \mathbf{MLP}_{\theta_d}(q^k, d_\lambda).
$$
We then calculate the KKT stationarity residual as follows

$$
r^{k+1}
= \nabla f_\lambda(\hat x^{k+1})
+ A_\lambda^\top z^{k+1}
+ \rho C_\lambda^\top(q^k - \hat s^{k+1}).
$$

During training, HUANet penalizes this residual so the learned primal update is not only feasible, but also closer to satisfying first-order optimality conditions.


In [11]:
class DualNet(nn.Module):
    n_eq:         int
    hidden_layers: tuple[int, ...]

    def setup(self) -> None:
        self.kernel_init = nn.initializers.xavier_uniform()
        self.bias_init   = nn.initializers.zeros_init()

    @nn.compact
    def __call__(self, nn_input: jnp.ndarray) -> jnp.ndarray:
        x = nn_input
        for features in self.hidden_layers:
            x = nn.Dense(features, kernel_init=self.kernel_init, bias_init=self.bias_init)(x)
            x = nn.gelu(x)
        return nn.Dense(self.n_eq, kernel_init=self.kernel_init, bias_init=self.bias_init)(x)

With the two subnetworks defined, `HUANet` connects `PrimeNet` and `DualNet` into one model. It first concatenates the ADMM input $q^k$ with the problem right-hand side $d_\lambda$, then sends this concatenated input to both subnetworks.


In [12]:
class HUANet(nn.Module):
    hidden_layers: tuple[int, ...]
    n_var:  int
    n_in: int
    n_eq: int
    mode: str = "correction"   # "correction" | "feasibility"

    def setup(self) -> None:
        self.prime_net = PrimeNet(hidden_layers=self.hidden_layers, n_var=self.n_var, n_in=self.n_in, mode=self.mode)
        self.dual_net  = DualNet(hidden_layers=self.hidden_layers, n_eq=self.n_eq)

    def __call__(self, q:jnp.ndarray, lam_param:jnp.ndarray, E:jnp.ndarray = None, EtE_inv:jnp.ndarray = None, eta:jnp.ndarray = None, return_z: bool = False) -> Tuple[jnp.ndarray, jnp.ndarray] | Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        nn_input = jnp.concatenate([q, lam_param], axis=-1)
        nn_z = self.dual_net(nn_input)

        if self.mode == "feasibility":
            # No KKT projection
            x_bar = self.prime_net(nn_input)
            x_hat = nn.softmax(x_bar)
            return x_hat, nn_z
        else:
            # Correction stage: KKT projection enforces constraints
            x_bar, s_bar = self.prime_net(nn_input)
            if eta.ndim == 2:
                in_axes = (0, 0, 0, 0, None, None)
            else:
                in_axes = (0, 0, 0, None, None, None)
            x_hat, s_hat = vmap(correction, in_axes=in_axes)(x_bar, s_bar, lam_param, eta, E, EtE_inv)

            if return_z:
                return x_hat, s_hat, nn_z
            return x_hat, s_hat

Now that we've designed structure for the neural network, let's implement $N$ HUANet layers as an unrolled ADMM trajectory via `unrolled_admm`.


In [13]:
def unrolled_admm(params, model, C:jnp.ndarray, lam_param:jnp.ndarray, batch_size: int, n_admm:int, n_var:int, n_in:int, rho:float, eps:float,gamma:float) -> Tuple[float, Tuple[float, float, float]]:
    nn_params = params["nn"]
    w_k = jnp.zeros((batch_size, n_in))
    v_k = jnp.zeros_like(w_k)
    s_k = jnp.zeros_like(w_k)

    def single_step(carry, _):
        w, v, _ = carry
        q_k = w - v / rho                              # (batch, n_in)

        x_hat, z_k = model.apply({"params": nn_params}, q_k, lam_param)

        # Slack: s = lam - C @ x
        s_hat = lam_param - x_hat @ C.T               # (batch, n_in)

        # Objective: f(x) = sum(x * log x)
        f_val = jnp.sum(x_hat * jnp.log(jnp.clip(x_hat, 1e-15)), axis=-1)   # (batch,)

        # KKT stationarity residual
        beta_k = -rho * (s_hat - q_k)
        grad_f = jnp.log(jnp.clip(x_hat, 1e-15)) + 1.0   # (batch, n_var)
        r1 = grad_f + (beta_k @ C) + z_k @ jnp.ones((z_k.shape[-1], n_var)) # (batch, n_var)
        r1_sq  = jnp.sum(r1 ** 2, axis=-1)                # (batch,)

        w_next = jnp.maximum(0.0, s_hat + v / rho)
        v_next = v + rho * (s_hat - w_next)

        return (w_next, v_next, s_hat), (f_val, r1_sq)

    (_, _, s_hat), (f_hist, r1_sq_hist) = jax.lax.scan(single_step, (w_k, v_k, s_k), None, length=n_admm, unroll=True)

    f_terminal  = jnp.mean(f_hist[-1])
    kkt_r1      = jnp.mean(jnp.sum(r1_sq_hist, axis=0) / n_var)
    kkt_last_r1 = jnp.mean(r1_sq_hist[-1]) / n_var
    viol        = jnp.mean(jnp.sum(jnp.maximum(0, -s_hat), axis=-1) / n_in)

    total_loss = f_terminal + eps * kkt_r1 + gamma * viol

    return total_loss, (f_terminal, kkt_last_r1, viol)


After defining the unrolled trajectory, `train_step` performs one gradient update of HUANet. It evaluates the unrolled loss ${L}(\theta)$,
then uses automatic differentiation to compute $\nabla_\theta {L}$ and updates the parameters with AdamW.
The JIT annotation compiles this update so repeated training epochs run faster.


In [14]:
@partial(jax.jit, static_argnames=["model", "optimizer", "batch_size", "n_admm", "n_var", "n_in"])
def train_step(params, opt_state, model, C:jnp.ndarray, lam_param:jnp.ndarray, batch_size:int, n_admm:int, n_var:int, n_in:int, rho:float, eps:float, gamma:float, optimizer) -> Tuple[jnp.ndarray, jnp.ndarray, float, float, float, float]:
    def loss_fn(all_params):
        return unrolled_admm(
            all_params, model,
            C, lam_param, batch_size, n_admm, n_var, n_in, rho, eps, gamma,
        )

    (loss, (f_terminal, kkt_last, viol)), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
    updates, opt_state_new = optimizer.update(grads, opt_state, params)
    params_new = optax.apply_updates(params, updates)

    return params_new, opt_state_new, loss, f_terminal, kkt_last, viol

For validation, `make_validator` builds a fixed-parameter HUANet solver. It runs the same ADMM recurrence as training,

$$
q^k \rightarrow \hat x^{k+1} \rightarrow \hat s^{k+1} \rightarrow (w^{k+1}, v^{k+1}),
$$

but without gradients or parameter updates. The output is the final predicted solution $\hat x^N$.


In [15]:
def make_validator(nn_params, model, C: jnp.ndarray, n_admm: int, rho: float):
    n_in = C.shape[0]

    @jax.jit
    def solve(lam_batch: jnp.ndarray) -> jnp.ndarray:
        n_samples = lam_batch.shape[0]
        w_init = jnp.zeros((n_samples, n_in))
        v_init = jnp.zeros_like(w_init)
        x_init = jnp.zeros((n_samples, model.n_var))

        def body_fn(carry, _):
            w_k, v_k, _ = carry
            q_k    = w_k - v_k / rho
            x_next, _ = model.apply({"params": nn_params}, q_k, lam_batch)
            s_next = lam_batch - x_next @ C.T
            w_next = jnp.maximum(0.0, s_next + v_k / rho)
            v_next = v_k + rho * (s_next - w_next)
            return (w_next, v_next, x_next), None

        (_, _, x_final), _ = jax.lax.scan(body_fn, (w_init, v_init, x_init), None, length=n_admm)
        return x_final

    return solve

Use this function to validate the trained HUANet model.


In [16]:
def validate(params, model, C, lam_val, batch_size, n_admm, rho):
    nn_params = params["nn"]
    solve = make_validator(nn_params, model, C, n_admm, rho)
    # warm up
    _ = solve(jnp.zeros((1, lam_val.shape[1]))).block_until_ready()

    all_x = []
    for start in range(0, len(lam_val), batch_size):
        lam_batch = jnp.array(lam_val[start : min(start + batch_size, len(lam_val))])
        x_hat = solve(lam_batch)
        x_hat.block_until_ready()
        all_x.append(np.array(x_hat))
    x_pred = np.concatenate(all_x, axis=0)

    lam_np = np.array(lam_val[: len(x_pred)])
    C_np   = np.array(C)

    x_safe = np.clip(x_pred, 1e-15, None)
    obj    = float(np.mean(np.sum(x_safe * np.log(x_safe), axis=1)))

    ineq_resid    = np.maximum(0, (x_pred @ C_np.T) - lam_np)
    ineq_viol_max = float(np.max(np.max(ineq_resid, axis=1)))

    eq_viol = np.abs(np.sum(x_pred, axis=1) - 1.0)
    eq_viol_max = float(np.max(eq_viol))

    return {
        "obj":         obj,
        "ineq_viol_max": ineq_viol_max,
        "eq_viol_max":   eq_viol_max,
    }

Finally, `print_val_summary` formats the validation dictionary into a compact report: objective value, maximum equality violation, and maximum inequality violation.


In [17]:
def print_val_summary(epoch, metrics):
    print(f"\nVALIDATION at epoch {epoch}")
    print("=" * 60)
    print(f"  Objective:           {metrics['obj']:.6e}")
    print(f"  Eq  violation max: {metrics['eq_viol_max']:.6e}")
    print(f"  Ineq violation max: {metrics['ineq_viol_max']:.6e}")
    print("=" * 60)

After the training utilities are defined, load the generated entropy dataset and define where the trained HUANet parameters will be saved.

In [18]:
dataset_path = Path(f"HUANet/data/datasets_demo_{n_samples}.npz")
model_path   = Path(f"HUANet/data/models_demo_{n_samples}.npz")
data         = np.load(dataset_path)

FileNotFoundError: [Errno 2] No such file or directory: 'HUANet/data/datasets_demo_2000.npz'

Next, split the generated $d_\lambda$ samples into training, validation, and test pools. Training updates the model, validation monitors generalization during training, and test data are reserved for final benchmarking.


In [ ]:
lam_all = data["lam"]          # (n_total, n_eq)
n_train = int(n_samples * train_percent)
n_val   = int(n_samples * val_percent)
n_test = n_samples - n_train - n_val

lam_train_pool = lam_all[:n_train]
lam_val_pool   = lam_all[n_train : n_train + n_val]
lam_test_pool  = lam_all[n_train + n_val : n_train + n_val + n_test]

With the data split ready, initialize HUANet in `feasibility` mode. In this mode, the model outputs $x$ through `softmax`, so

$$
x_i > 0, \qquad \mathbf{1}^\top x = 1.
$$

The cell also initializes AdamW and the learning-rate schedule used to optimize the HUANet parameters.


In [ ]:
hidden_layers = tuple(hidden_layers)
model = HUANet(hidden_layers=hidden_layers, n_var=n_var, n_in=n_in, n_eq=n_eq, mode="feasibility")
key   = jax.random.PRNGKey(seed)

nn_params = model.init(key, jnp.zeros((1, n_in)), jnp.zeros((1, n_in)))["params"]

params = {"nn": nn_params}

lr_sched = optax.exponential_decay(
    init_value=1e-3,
    decay_rate=0.9,
    transition_steps=1000,
)
optimizer = optax.adamw(learning_rate=lr_sched, weight_decay=1e-4)
opt_state = optimizer.init(params)

Now we run the training loop. Each epoch samples a mini-batch of $d_\lambda$, evaluates the unrolled loss, updates HUANet parameters, and keeps the best parameter set observed so far.
Periodic validation checks whether the learned solver remains feasible and stable on held-out problem instances.


In [ ]:
print("=" * 80)
print("Training Entropy — HUANet (feasibility mode)")
print("=" * 80)
rng        = np.random.default_rng(seed)
best_loss  = float("inf")
best_params = params

for epoch in range(n_epochs):
    batch_idx  = rng.integers(0, n_train, size=batch_size)
    lam_param  = jnp.array(lam_train_pool[batch_idx])

    params, opt_state, loss, f_terminal, kkt_last, viol = train_step(
        params, opt_state, model,
        C, lam_param, batch_size, n_admm, n_var, n_in, rho, eps, gamma, optimizer,
    )

    if float(loss)  < best_loss:
        best_loss   = float(loss)
        best_params = params

    if epoch % 50 == 0:
        current_lr = float(lr_sched(epoch))
        print(
            f"Epoch {epoch:5d} | loss: {float(loss):.6f} | "
            f"f: {float(f_terminal):.6f} | "
            f"kkt: {float(kkt_last):.6f} | "
            f"viol: {float(viol):.6f} | "
            f"lr: {current_lr:.2e}"
        )

    if epoch % 100 == 0:
        val_metrics = validate(best_params, model, C, lam_val_pool, batch_size, n_admm, rho)
        print_val_summary(epoch, val_metrics)

val_metrics = validate(best_params, model, C, lam_val_pool, batch_size, n_admm, rho)
print_val_summary(n_epochs, val_metrics)


The training line reports:

- **loss:** the loss function $L$.
- **f:** the final entropy objective.
- **kkt:** the KKT residual term. Smaller values indicate that the learned update is closer to satisfying first-order optimality conditions.
- **viol:** the inequality-violation penalty.
- **lr:** the learning rate at that epoch.

The training process shows decreasing loss, small validation equality violation, and small validation inequality violation.


After training, save the best HUANet parameters and metadata.

In [ ]:
np.savez_compressed(
    model_path,
    params=jax.tree_util.tree_map(np.array, best_params),
    best_loss=float(best_loss),
    n_epochs=n_epochs,
    n_train=n_train,
    n_val=n_val,
    n_test=n_test,
    scenario_n_var=n_var,
    hidden_layers=np.array(hidden_layers),
)
print(f"Saved trained model to {model_path}")

# Benchmark results

After training, we evaluate HUANet on test problem instances.
In the first benchmark, using [Clarabel](https://clarabel.org/stable/) as ground truth, we report: Optimality gap and Feasibility.
We also compare **solve time** among HUANet, Clarabel, and [SCS](https://www.cvxgrp.org/scs/) to show how fast the learned solver is relative to classical solvers.
After that, we compare runtime between learned unrolled ADMM and classical ADMM.


To begin benchmarking, `make_l2o_solver` builds the trained HUANet inference solver.
The implementation uses `pmap` so batches can be sharded across available JAX devices.


In [ ]:
def make_l2o_solver(nn_params, model: HUANet, C: jnp.ndarray, n_var: int, n_in: int, n_admm: int, rho: float):
    @jax.pmap
    def solve(lam_batch: jnp.ndarray) -> jnp.ndarray:
        n_samples = lam_batch.shape[0]

        w_init = jnp.zeros((n_samples, n_in))
        v_init = jnp.zeros_like(w_init)
        x_init = jnp.zeros((n_samples, n_var))

        def body_fn(_, carry):
            w_k, v_k, _ = carry
            q_k    = w_k - v_k / rho
            x_next, _ = model.apply({"params": nn_params}, q_k, lam_batch)
            s_next = lam_batch - x_next @ C.T
            w_next = jnp.maximum(0.0, s_next + v_k / rho)
            v_next = v_k + rho * (s_next - w_next)
            return w_next, v_next, x_next

        _, _, x_final = jax.lax.fori_loop(0, n_admm, body_fn, (w_init, v_init, x_init), unroll=True)
        return x_final

    return solve

For comparison, `baseline_solver` solves the original entropy problem with [CVXPY](https://www.cvxpy.org/).

In [ ]:
def baseline_solver(lam_samples: np.ndarray, C: np.ndarray, n_var: int, solver_name: str) -> Tuple[np.ndarray, np.ndarray]:
    x_batch, solve_times = [], []
    for lam_sample in lam_samples:
        x = cp.Variable(n_var)
        problem = cp.Problem(cp.Minimize(cp.sum(cp.entr(x) * -1)),
            [C @ x <= lam_sample, cp.sum(x) == 1],
        )
        problem.solve(solver=solver_name, warm_start=True, verbose=False)
        x_batch.append(np.asarray(x.value, dtype=float))
        solve_times.append(problem.solver_stats.solve_time)

    return np.array(x_batch), np.array(solve_times, dtype=float)

Using the solver outputs, `benchmark_metrics` compares HUANet predictions with reference solver solutions. It reports objective gap, equality violation, inequality violation, and solve time.

In [ ]:
def benchmark_metrics(x_pred:np.ndarray, x_true:np.ndarray, C:np.ndarray, lam_test:np.ndarray, run_time:np.ndarray) -> dict:
    # Objective: f(x) = sum(x * log x)
    x_safe_pred = np.clip(x_pred, 1e-15, None)
    x_safe_true = np.clip(x_true, 1e-15, None)
    obj_pred = np.sum(x_safe_pred * np.log(x_safe_pred), axis=1)
    obj_true = np.sum(x_safe_true * np.log(x_safe_true), axis=1)

    gap_percent = np.abs((obj_pred - obj_true) / np.maximum(np.abs(obj_true), 1e-10)) * 100

    # Inequality: Cx <= lam
    ineq_resid = np.maximum(0, (x_pred @ C.T) - lam_test)   # (n_samples, n_ineq)
    ineq_viol  = np.max(ineq_resid, axis=1)

    # Equality: 1^T x = 1
    eq_viol = np.abs(np.sum(x_pred, axis=1) - 1.0)

    return {
        "mean_obj_gap": np.mean(gap_percent),
        "max_obj_gap": np.max(gap_percent),
        "mean_ineq_viol": np.mean(ineq_viol),
        "max_ineq_viol": np.max(ineq_viol),
        "mean_eq_viol": np.mean(eq_viol),
        "max_eq_viol": np.max(eq_viol),
        "mean_solve_time": np.mean(run_time),
        "max_solve_time": np.max(run_time),
    }

Next, run the main benchmark. The trained HUANet model is loaded, evaluated on the test set, and compared with Clarabel and SCS on the same $d_\lambda$ instances.
The warm-up call removes JAX compilation time from the measured inference times.
For HUANet, the code uses **per-batch timing**. The test samples are padded and sharded across the available JAX devices. For each sharded batch, the code records wall-clock time around one `l2o_solve` call and uses `block_until_ready()` so asynchronous JAX execution finishes before the timer stops.

The elapsed batch time is then assigned to the samples in that batch, giving a per-instance solve-time array `l2o_times`.


In [ ]:
model_path   = Path(f"HUANet/data/models_demo_2000.npz")
# -- Load trained model --
saved            = np.load(model_path, allow_pickle=True)
saved_params_obj = saved["params"].item()
hidden_layers    = tuple(saved["hidden_layers"])
n_train          = int(saved["n_train"])
n_val            = int(saved["n_val"])

# -- Build model --
model     = HUANet(hidden_layers=hidden_layers, n_var=n_var, n_in=n_in, n_eq=n_eq, mode="feasibility")
nn_params = saved_params_obj["nn"]

# ---- HUANet ----
l2o_solve = make_l2o_solver(nn_params=nn_params, model=model, C=C, n_var=n_var, n_in=n_in, n_admm=n_admm, rho=rho)
n_devices = jax.local_device_count()
print(f"Using {n_devices} CPU devices via pmap")
pad = (n_devices - len(lam_test_pool) % n_devices) % n_devices
if pad > 0:
    lam_padded = jnp.concatenate(
        [lam_test_pool, jnp.zeros((pad, n_in), dtype=lam_test_pool.dtype)], axis=0
    )
else:
    lam_padded = lam_test_pool
per_device = len(lam_padded) // n_devices

# Warm up
lam_warm = jnp.zeros((n_devices, 1, n_in), dtype=lam_test_pool.dtype)
_ = l2o_solve(lam_warm).block_until_ready()

# Per-batch timing
l2o_times  = []
x_pred_list = []
for i in range(per_device):
    lam_slice         = lam_padded[i * n_devices:(i + 1) * n_devices]   # (n_devices, n_in)
    lam_slice_sharded = lam_slice[:, None, :]                            # (n_devices, 1, n_in)
    start   = time.perf_counter()
    x_slice = l2o_solve(lam_slice_sharded)
    x_slice.block_until_ready()
    elapsed = time.perf_counter() - start
    x_pred_list.append(np.array(x_slice[:, 0, :]))                      # (n_devices, n_var)
    l2o_times.extend([elapsed / n_devices] * n_devices)

x_pred_array = np.vstack(x_pred_list)[:len(lam_test_pool)]
l2o_times    = np.array(l2o_times[:len(lam_test_pool)], dtype=float)
sequential_time = float(np.mean(l2o_times))
C_np = np.array(C)

In [ ]:
print(f"Running L2O...")
# ---- Clarabel ----
print("Running Clarabel...")
x_clarabel, solve_times_clarabel = baseline_solver(lam_test_pool, C_np, n_var, cp.CLARABEL)
# ---- SCS ----
print("Running SCS...")
x_scs, solve_times_scs = baseline_solver(lam_test_pool, C_np, n_var, "SCS")

After running all solvers, print a compact metric table for the learned solver. Each metric is shown in `(mean)max` format, so the table reports both typical behavior and worst-case behavior over the test set.


In [ ]:
l2o_metrics = benchmark_metrics(x_pred_array, x_clarabel, C_np, np.array(lam_test_pool), l2o_times)
# metric_keys = {
#     "mean_obj_gap", "max_obj_gap",
#     "mean_ineq_viol", "max_ineq_viol",
#     "mean_eq_viol", "max_eq_viol",
#     "mean_solve_time", "max_solve_time",
# }

def mean_max(metrics, mean_key, max_key):
    return f"({metrics[mean_key]:.3e}){metrics[max_key]:.3e}"

headers = [
    "Metrics",
    "n_var",
    "(n_eq, n_in)",
    "Optimality gap (%)",
    "Eq.violation",
    "Ineq.violation",
    "Solve time (s)",
]
row = [
    "Entropy (L2O solver)",
    str(n_var),
    f"({n_eq}, {n_in})",
    mean_max(l2o_metrics, "mean_obj_gap", "max_obj_gap"),
    mean_max(l2o_metrics, "mean_eq_viol", "max_eq_viol"),
    mean_max(l2o_metrics, "mean_ineq_viol", "max_ineq_viol"),
    mean_max(l2o_metrics, "mean_solve_time", "max_solve_time"),
]

widths = [max(len(header), len(value)) for header, value in zip(headers, row)]
header_line = "| " + " | ".join(header.ljust(width) for header, width in zip(headers, widths)) + " |"
divider = "| " + " | ".join("-" * width for width in widths) + " |"
row_line = "| " + " | ".join(value.ljust(width) for value, width in zip(row, widths)) + " |"

print(header_line)
print(divider)
print(row_line)

After the table, plot the solve-time distributions for HUANet, Clarabel, and SCS.

In [ ]:
times_ms = [
    np.asarray(l2o_times) * 1e3,
    np.asarray(solve_times_clarabel) * 1e3,
    np.asarray(solve_times_scs) * 1e3,
]
tick_labels = ["HUANet", "Clarabel", "SCS"]
colors = ["#FF2020", "#4477AA", "#44AA77"]

fig, ax = plt.subplots(figsize=(5.0, 4.0))
box = ax.boxplot(times_ms, tick_labels=tick_labels, showfliers=False, patch_artist=True)
for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_ylabel("Solve time (ms)")
ax.set_title("Entropy maximization")
ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


We implement vanilla ADMM for the entropy problem.

In [ ]:
def primal_solver(lam_samples: np.ndarray, C: np.ndarray, qk: np.ndarray, rho: float, solver_name=cp.CLARABEL) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    n_var = C.shape[1]
    n_eq  = C.shape[0]
    x_batch, s_batch, solve_times = [], [], []
    for i, lam_sample in enumerate(lam_samples):
        x_var = cp.Variable(n_var)
        s_var = cp.Variable(n_eq)
        problem = cp.Problem(cp.Minimize(cp.sum(cp.entr(x_var) * -1)+ 0.5 * rho * cp.sum_squares(s_var - qk[i])),
            [C @ x_var + s_var == lam_sample, cp.sum(x_var) == 1, x_var >= 0],
        )
        problem.solve(solver=solver_name, warm_start=True, verbose=False)
        x_batch.append(x_var.value)
        s_batch.append(s_var.value)
        solve_times.append(problem.solver_stats.solve_time)
    return np.array(x_batch), np.array(s_batch), np.array(solve_times, dtype=float)


def admm(C: np.ndarray, lam_batch: np.ndarray, max_iter: int, tol: float, rho: float) -> Tuple[np.ndarray, np.ndarray]:
    n_samples = lam_batch.shape[0]
    n_var = C.shape[1]
    n_eq  = C.shape[0]

    x_batch          = np.zeros((n_samples, n_var))
    per_sample_times = np.zeros(n_samples)
    per_sample_iters = np.zeros(n_samples, dtype=int)

    for i in range(n_samples):
        lam_i = lam_batch[i : i + 1]   # (1, n_eq)
        w_k   = np.zeros((1, n_eq))
        v_k   = np.zeros_like(w_k)
        x_k   = np.zeros((1, n_var))

        primal_res  = np.inf
        dual_res    = np.inf
        iter_count  = 0
        primal_time = 0.0
        overhead_time = 0.0

        for iter_count in range(1, max_iter + 1):
            t0 = time.perf_counter()
            q_k = w_k - v_k / rho
            overhead_time += time.perf_counter() - t0

            x_next, s_next, iter_times = primal_solver(lam_i, C, q_k, rho)
            primal_time += iter_times[0]

            t0 = time.perf_counter()
            w_next = np.maximum(0.0, s_next + v_k / rho)
            v_next = v_k + rho * (s_next - w_next)

            # Primal residual: ||s - w||_inf
            primal_res = np.max(np.abs(s_next - w_next))
            # Dual residual: rho * ||w_next - w_k||_inf
            dual_res = rho * np.max(np.abs(w_next - w_k))

            w_k, v_k, x_k = w_next, v_next, x_next
            overhead_time += time.perf_counter() - t0

            if primal_res <= tol and dual_res <= tol:
                break

        per_sample_times[i] = overhead_time + primal_time
        x_batch[i] = x_k[0]
        per_sample_iters[i] = iter_count

    return x_batch, per_sample_times

Let's run the direct ADMM baseline on the test set.


In [ ]:
max_iter = 1000  # number of iterations
tol = 1e-4       # termination threshold

print(f"\nRunning ADMM on {len(lam_test_pool)} samples...")
x_admm, times_admm = admm(C_np, np.array(lam_test_pool), max_iter, tol, rho)

Finally, plot the mean solve time of HUANet and direct ADMM.

In [ ]:
labels = ["HUANet", "ADMM"]
mean_times_ms = [np.mean(l2o_times) * 1e3, np.mean(times_admm) * 1e3]
colors = ["#FF2020", "#006400"]

fig, ax = plt.subplots(figsize=(4.0, 5.0))
bars = ax.bar(labels, mean_times_ms, color=colors, edgecolor="black", linewidth=0.6, alpha=0.85)

for bar, value in zip(bars, mean_times_ms):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value * 1.05,
        f"{value:.3g} ms",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_ylabel("Mean solve time (ms)")
ax.set_title(f"Entropy maximization")
ax.set_yscale("log")
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
